# 10 Baseline Threshold Classifier

## Purpose

This notebook builds the first baseline probability model for Hong Kong temperature-threshold outcomes.

The previous threshold-classification dataset defined the supervised-learning target:

```math
Y_t^{(K)} = \mathbf{1}\{T_t^{\mathrm{settlement}} \geq K\}.
```

This notebook trains a simple baseline classifier using official Hong Kong Observatory historical daily maximum temperature data. The model is intentionally simple and should be interpreted as a benchmark rather than the final dissertation model. Its purpose is to establish the supervised-learning workflow before AI weather forecast features are merged.

The output is a baseline probability for each threshold in the Hong Kong Polymarket example. Later notebooks can compare this baseline with market-implied probabilities and with AI-weather post-processing models.

## 1. Imports and paths

In [20]:
from pathlib import Path
from datetime import datetime, timezone
from io import StringIO
import re
import warnings

import requests
import pandas as pd
import numpy as np

try:
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
    SKLEARN_AVAILABLE = True
except Exception as e:
    SKLEARN_AVAILABLE = False
    SKLEARN_IMPORT_ERROR = repr(e)

RAW_DIR = Path("../data/raw/baseline_classifier")
PROCESSED_DIR = Path("../data/processed/baseline_classifier")

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_rows", 120)

print("Notebook run time UTC:", datetime.now(timezone.utc).isoformat())
print("Raw directory:", RAW_DIR)
print("Processed directory:", PROCESSED_DIR)
print("scikit-learn available:", SKLEARN_AVAILABLE)

if not SKLEARN_AVAILABLE:
    print("scikit-learn import error:", SKLEARN_IMPORT_ERROR)

Notebook run time UTC: 2026-06-17T05:01:56.905848+00:00
Raw directory: ../data/raw/baseline_classifier
Processed directory: ../data/processed/baseline_classifier
scikit-learn available: True


## 2. Configuration

The first baseline is trained for Hong Kong only. The threshold grid is loaded from the Step 7 dataset if available. Otherwise, the default grid is set to the thresholds used in the Hong Kong Polymarket example.

In [23]:
CONFIG = {
    "city": "Hong Kong",
    "station": "HKO",
    "settlement_source": "Hong Kong Observatory",
    "settlement_variable": "Daily maximum temperature",
    "training_start_year": 2010,
    "validation_years": [2024, 2025],
    "test_years": [2026],
    "default_threshold_grid_c": list(range(24, 34)),
    "step7_threshold_dataset_path": Path("../data/processed/threshold_dataset/hong_kong_threshold_classification_dataset.csv"),
}

CONFIG

{'city': 'Hong Kong',
 'station': 'HKO',
 'settlement_source': 'Hong Kong Observatory',
 'settlement_variable': 'Daily maximum temperature',
 'training_start_year': 2010,
 'validation_years': [2024, 2025],
 'test_years': [2026],
 'default_threshold_grid_c': [24, 25, 26, 27, 28, 29, 30, 31, 32, 33],
 'step7_threshold_dataset_path': PosixPath('../data/processed/threshold_dataset/hong_kong_threshold_classification_dataset.csv')}

## 3. Helper functions

In [26]:
def normalise_col(col):
    return re.sub(r"[^a-z0-9]+", "_", str(col).strip().lower()).strip("_")


def fetch_text(url, timeout=30):
    try:
        response = requests.get(url, timeout=timeout)
        return {
            "url": url,
            "status_code": response.status_code,
            "ok": response.ok,
            "content_type": response.headers.get("content-type"),
            "text": response.text,
            "error": None,
        }
    except Exception as e:
        return {
            "url": url,
            "status_code": None,
            "ok": False,
            "content_type": None,
            "text": "",
            "error": repr(e),
        }


def standardise_hko_daily_max(df):
    out = df.copy()
    out.columns = [normalise_col(c) for c in out.columns]

    required_ymd = {"year", "month", "day"}
    if required_ymd.issubset(set(out.columns)):
        date_series = pd.to_datetime(
            {
                "year": pd.to_numeric(out["year"], errors="coerce"),
                "month": pd.to_numeric(out["month"], errors="coerce"),
                "day": pd.to_numeric(out["day"], errors="coerce"),
            },
            errors="coerce",
        )
    else:
        date_candidates = [c for c in out.columns if c in ["date", "yyyymmdd"] or "date" in c]
        if date_candidates:
            date_series = pd.to_datetime(out[date_candidates[0]], errors="coerce")
        else:
            raise ValueError("No usable date fields found in HKO data.")

    temp_candidates = [
        c for c in out.columns
        if ("max" in c and ("temp" in c or "temperature" in c))
        or ("maximum" in c and ("temp" in c or "temperature" in c))
        or c in ["value", "temperature", "temp"]
    ]

    if temp_candidates:
        temp_col = temp_candidates[0]
    else:
        numeric_scores = {}
        for c in out.columns:
            if c in ["year", "month", "day"]:
                continue
            numeric_scores[c] = pd.to_numeric(out[c], errors="coerce").notna().sum()
        temp_col = max(numeric_scores, key=numeric_scores.get)

    standardised = pd.DataFrame({
        "date": date_series,
        "official_daily_max_temp_c": pd.to_numeric(out[temp_col], errors="coerce"),
    })

    standardised = standardised.dropna(subset=["date", "official_daily_max_temp_c"]).copy()
    standardised["date"] = pd.to_datetime(standardised["date"])
    standardised["year"] = standardised["date"].dt.year
    standardised["month"] = standardised["date"].dt.month
    standardised["day"] = standardised["date"].dt.day
    standardised["day_of_year"] = standardised["date"].dt.dayofyear
    standardised["station"] = CONFIG["station"]
    standardised["settlement_source"] = CONFIG["settlement_source"]

    return standardised.sort_values("date").reset_index(drop=True)


def add_calendar_features(df):
    out = df.copy()
    out["day_of_year_sin"] = np.sin(2 * np.pi * out["day_of_year"] / 366.0)
    out["day_of_year_cos"] = np.cos(2 * np.pi * out["day_of_year"] / 366.0)
    out["month_sin"] = np.sin(2 * np.pi * out["month"] / 12.0)
    out["month_cos"] = np.cos(2 * np.pi * out["month"] / 12.0)
    return out


def safe_log_loss(y_true, y_prob):
    labels = [0, 1]
    prob = np.clip(np.asarray(y_prob, dtype=float), 1e-6, 1 - 1e-6)
    return log_loss(y_true, prob, labels=labels)


def evaluate_binary_probabilities(df, target_col, prob_col, label):
    tmp = df[[target_col, prob_col]].dropna().copy()

    if tmp.empty:
        return {
            "sample": label,
            "n_obs": 0,
            "positive_rate": np.nan,
            "brier_score": np.nan,
            "log_loss": np.nan,
            "roc_auc": np.nan,
        }

    y = tmp[target_col].astype(int)
    p = np.clip(tmp[prob_col].astype(float), 1e-6, 1 - 1e-6)

    if y.nunique() > 1:
        try:
            auc = roc_auc_score(y, p)
        except Exception:
            auc = np.nan
    else:
        auc = np.nan

    return {
        "sample": label,
        "n_obs": len(tmp),
        "positive_rate": y.mean(),
        "brier_score": brier_score_loss(y, p),
        "log_loss": safe_log_loss(y, p),
        "roc_auc": auc,
    }

## 4. Load official HKO historical daily maximum temperature

The classifier is trained on official HKO historical daily maximum temperature data. This keeps the target source consistent with the Hong Kong settlement-source work.

In [29]:
hko_url = "https://data.weather.gov.hk/weatherAPI/opendata/opendata.php?dataType=CLMMAXT&rformat=csv&station=HKO"

result = fetch_text(hko_url)

print("HKO request status:", result["status_code"])
print("Content type:", result["content_type"])
print("Characters returned:", len(result["text"]))

if not result["ok"] or not result["text"].strip():
    raise RuntimeError(f"HKO open data retrieval failed: {result['error']}")

raw_path = RAW_DIR / "hko_daily_max_temperature.csv"
raw_path.write_text(result["text"], encoding="utf-8")


def read_hko_csv_flexibly(csv_text, max_skiprows=30):
    """
    HKO CSV files may contain metadata lines before the real table.
    This function searches for the first usable table header and then reads
    the CSV from that point.
    """
    lines = csv_text.splitlines()

    # First preference: find a header row containing Year, Month and Day.
    for i, line in enumerate(lines[:max_skiprows + 1]):
        normalised = line.lower().replace(" ", "")
        if "year" in normalised and "month" in normalised and "day" in normalised:
            df = pd.read_csv(StringIO(csv_text), skiprows=i)
            return df, i

    # Fallback: try different skiprow values and keep the first plausible table.
    for skiprows in range(max_skiprows + 1):
        try:
            df = pd.read_csv(StringIO(csv_text), skiprows=skiprows)
            cols = [normalise_col(c) for c in df.columns]

            has_date_cols = {"year", "month", "day"}.issubset(set(cols))
            has_enough_rows = df.shape[0] > 100
            has_enough_cols = df.shape[1] >= 3

            if has_date_cols or (has_enough_rows and has_enough_cols):
                return df, skiprows

        except Exception:
            continue

    raise ValueError("Could not parse HKO CSV into a usable table.")


hko_raw, skiprows_used = read_hko_csv_flexibly(result["text"])

print("CSV parsed with skiprows:", skiprows_used)
print("Raw parsed shape:", hko_raw.shape)
display(hko_raw.head())

hko_daily_df = standardise_hko_daily_max(hko_raw)
hko_daily_df = add_calendar_features(hko_daily_df)

hko_daily_df = hko_daily_df[hko_daily_df["year"] >= CONFIG["training_start_year"]].copy()

print("HKO daily data shape:", hko_daily_df.shape)
print("Date range:", hko_daily_df["date"].min().date(), "to", hko_daily_df["date"].max().date())

display(hko_daily_df.head())
display(hko_daily_df.tail())

HKO request status: 200
Content type: text/csv; charset=utf-8
Characters returned: 838759
CSV parsed with skiprows: 2
Raw parsed shape: (49463, 5)


,年/Year,月/Month,日/Day,數值/Value,數據完整性/data Completeness
0,1884,1.0,1.0,15.3,C
1,1884,1.0,2.0,17.1,C
2,1884,1.0,3.0,19.6,C
3,1884,1.0,4.0,23.2,C
4,1884,1.0,5.0,19.4,C


HKO daily data shape: (5995, 12)
Date range: 2010-01-01 to 2026-05-31


,date,official_daily_max_temp_c,year,month,day,day_of_year,station,settlement_source,day_of_year_sin,day_of_year_cos,month_sin,month_cos
43464,2010-01-01,18.1,2010,1,1,1,HKO,Hong Kong Observatory,0.017166,0.999853,0.5,0.866025
43465,2010-01-02,20.0,2010,1,2,2,HKO,Hong Kong Observatory,0.034328,0.999411,0.5,0.866025
43466,2010-01-03,18.3,2010,1,3,3,HKO,Hong Kong Observatory,0.051479,0.998674,0.5,0.866025
43467,2010-01-04,19.9,2010,1,4,4,HKO,Hong Kong Observatory,0.068615,0.997643,0.5,0.866025
43468,2010-01-05,18.5,2010,1,5,5,HKO,Hong Kong Observatory,0.085731,0.996318,0.5,0.866025


,date,official_daily_max_temp_c,year,month,day,day_of_year,station,settlement_source,day_of_year_sin,day_of_year_cos,month_sin,month_cos
49454,2026-05-27,33.7,2026,5,27,147,HKO,Hong Kong Observatory,0.579421,-0.815028,0.5,-0.866025
49455,2026-05-28,33.4,2026,5,28,148,HKO,Hong Kong Observatory,0.565345,-0.824855,0.5,-0.866025
49456,2026-05-29,34.1,2026,5,29,149,HKO,Hong Kong Observatory,0.551102,-0.834438,0.5,-0.866025
49457,2026-05-30,32.6,2026,5,30,150,HKO,Hong Kong Observatory,0.536696,-0.843776,0.5,-0.866025
49458,2026-05-31,30.9,2026,5,31,151,HKO,Hong Kong Observatory,0.522133,-0.852864,0.5,-0.866025


## 5. Load the threshold grid

The threshold grid is taken from the Step 7 dataset where available. If the Step 7 output is not present locally, the default Hong Kong threshold grid is used.

In [32]:
step7_path = CONFIG["step7_threshold_dataset_path"]

if step7_path.exists():
    step7_threshold_df = pd.read_csv(step7_path)
    threshold_grid = sorted(step7_threshold_df["threshold_c"].dropna().unique().astype(float).tolist())
    print("Loaded threshold grid from Step 7 output.")
else:
    step7_threshold_df = pd.DataFrame()
    threshold_grid = [float(x) for x in CONFIG["default_threshold_grid_c"]]
    print("Step 7 output not found. Using default threshold grid.")

print("Threshold grid:", threshold_grid)

if not step7_threshold_df.empty:
    display(step7_threshold_df.head())

Loaded threshold grid from Step 7 output.
Threshold grid: [24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0]


,contract_slug,city,target_local_date,settlement_source,settlement_variable,official_realised_temperature_c,threshold_c,realised_exceeds_threshold,forecast_model,forecast_run_time_utc,forecast_issue_time_utc,forecast_valid_time_utc,timing_label,forecast_timing_category,lead_time_to_valid_hours,lead_time_to_day_start_hours,lead_time_to_day_midpoint_hours,lead_time_to_day_end_hours,market_implied_threshold_prob_raw,market_implied_threshold_prob_normalised,number_of_contributing_bins,average_price_time_gap_minutes,market_price_timestamp_rule,forecast_temperature_c,forecast_temperature_source,forecast_feature_status
0,highest-temperature-in-hong-kong-on-may-30-2026,Hong Kong,2026-05-30,Hong Kong Observatory,Daily Maximum Temperature,32.6,24.0,1,AIFS_or_ECMWF_example,2026-05-28 12:00:00+00:00,2026-05-28 12:00:00+00:00,2026-05-30 12:00:00+00:00,two_day_valid_time_example,full_day_ex_ante,48.0,28.0,40.0,51.999722,0.9645,0.998964,10,0.1,first available Polymarket price at or after f...,NaN,not_attached_in_step_7,pending_ai_weather_feature_merge
1,highest-temperature-in-hong-kong-on-may-30-2026,Hong Kong,2026-05-30,Hong Kong Observatory,Daily Maximum Temperature,32.6,25.0,1,AIFS_or_ECMWF_example,2026-05-28 12:00:00+00:00,2026-05-28 12:00:00+00:00,2026-05-30 12:00:00+00:00,two_day_valid_time_example,full_day_ex_ante,48.0,28.0,40.0,51.999722,0.9620,0.996375,9,0.1,first available Polymarket price at or after f...,NaN,not_attached_in_step_7,pending_ai_weather_feature_merge
2,highest-temperature-in-hong-kong-on-may-30-2026,Hong Kong,2026-05-30,Hong Kong Observatory,Daily Maximum Temperature,32.6,26.0,1,AIFS_or_ECMWF_example,2026-05-28 12:00:00+00:00,2026-05-28 12:00:00+00:00,2026-05-30 12:00:00+00:00,two_day_valid_time_example,full_day_ex_ante,48.0,28.0,40.0,51.999722,0.9595,0.993786,8,0.1,first available Polymarket price at or after f...,NaN,not_attached_in_step_7,pending_ai_weather_feature_merge
3,highest-temperature-in-hong-kong-on-may-30-2026,Hong Kong,2026-05-30,Hong Kong Observatory,Daily Maximum Temperature,32.6,27.0,1,AIFS_or_ECMWF_example,2026-05-28 12:00:00+00:00,2026-05-28 12:00:00+00:00,2026-05-30 12:00:00+00:00,two_day_valid_time_example,full_day_ex_ante,48.0,28.0,40.0,51.999722,0.9540,0.988089,7,0.1,first available Polymarket price at or after f...,NaN,not_attached_in_step_7,pending_ai_weather_feature_merge
4,highest-temperature-in-hong-kong-on-may-30-2026,Hong Kong,2026-05-30,Hong Kong Observatory,Daily Maximum Temperature,32.6,28.0,1,AIFS_or_ECMWF_example,2026-05-28 12:00:00+00:00,2026-05-28 12:00:00+00:00,2026-05-30 12:00:00+00:00,two_day_valid_time_example,full_day_ex_ante,48.0,28.0,40.0,51.999722,0.9505,0.984464,6,0.1,first available Polymarket price at or after f...,NaN,not_attached_in_step_7,pending_ai_weather_feature_merge


## 6. Build the historical threshold-classification training table

Each historical date is expanded across all thresholds. The binary target equals one when the official daily maximum temperature is at least the threshold.

In [35]:
training_rows = []

for _, row in hko_daily_df.iterrows():
    for threshold_c in threshold_grid:
        training_rows.append({
            "city": CONFIG["city"],
            "date": row["date"],
            "year": int(row["year"]),
            "month": int(row["month"]),
            "day": int(row["day"]),
            "day_of_year": int(row["day_of_year"]),
            "day_of_year_sin": row["day_of_year_sin"],
            "day_of_year_cos": row["day_of_year_cos"],
            "month_sin": row["month_sin"],
            "month_cos": row["month_cos"],
            "threshold_c": float(threshold_c),
            "official_daily_max_temp_c": row["official_daily_max_temp_c"],
            "realised_exceeds_threshold": int(row["official_daily_max_temp_c"] >= threshold_c),
        })

historical_threshold_df = pd.DataFrame(training_rows)

historical_threshold_df["sample_split"] = np.select(
    [
        historical_threshold_df["year"].isin(CONFIG["validation_years"]),
        historical_threshold_df["year"].isin(CONFIG["test_years"]),
    ],
    [
        "validation",
        "test",
    ],
    default="train",
)

print("Historical threshold dataset shape:", historical_threshold_df.shape)

display(
    historical_threshold_df.groupby("sample_split")
    .agg(
        rows=("realised_exceeds_threshold", "size"),
        dates=("date", "nunique"),
        positive_rate=("realised_exceeds_threshold", "mean"),
    )
    .reset_index()
)

display(historical_threshold_df.head())

Historical threshold dataset shape: (59950, 14)


,sample_split,rows,dates,positive_rate
0,test,1510,151,0.263576
1,train,51130,5113,0.417524
2,validation,7310,731,0.458550


,city,date,year,month,day,day_of_year,day_of_year_sin,day_of_year_cos,month_sin,month_cos,threshold_c,official_daily_max_temp_c,realised_exceeds_threshold,sample_split
0,Hong Kong,2010-01-01,2010,1,1,1,0.017166,0.999853,0.5,0.866025,24.0,18.1,0,train
1,Hong Kong,2010-01-01,2010,1,1,1,0.017166,0.999853,0.5,0.866025,25.0,18.1,0,train
2,Hong Kong,2010-01-01,2010,1,1,1,0.017166,0.999853,0.5,0.866025,26.0,18.1,0,train
3,Hong Kong,2010-01-01,2010,1,1,1,0.017166,0.999853,0.5,0.866025,27.0,18.1,0,train
4,Hong Kong,2010-01-01,2010,1,1,1,0.017166,0.999853,0.5,0.866025,28.0,18.1,0,train


## 7. Train a baseline classifier

The preferred baseline is a logistic regression using calendar features and the threshold. If scikit-learn is not available, the notebook falls back to a smoothed climatological probability by month and threshold.

This model does not yet use AI weather forecasts. It is a benchmark that captures seasonal threshold-exceedance patterns.

In [38]:
feature_cols = [
    "threshold_c",
    "day_of_year_sin",
    "day_of_year_cos",
    "month_sin",
    "month_cos",
]

target_col = "realised_exceeds_threshold"

train_df = historical_threshold_df[historical_threshold_df["sample_split"] == "train"].copy()
validation_df = historical_threshold_df[historical_threshold_df["sample_split"] == "validation"].copy()
test_df = historical_threshold_df[historical_threshold_df["sample_split"] == "test"].copy()

model_metadata = {
    "model_name": None,
    "feature_cols": feature_cols,
    "training_start_year": CONFIG["training_start_year"],
    "validation_years": CONFIG["validation_years"],
    "test_years": CONFIG["test_years"],
}

if SKLEARN_AVAILABLE:
    baseline_model = Pipeline([
        ("scaler", StandardScaler()),
        ("logistic", LogisticRegression(max_iter=1000)),
    ])

    baseline_model.fit(train_df[feature_cols], train_df[target_col])

    historical_threshold_df["baseline_classifier_probability"] = baseline_model.predict_proba(
        historical_threshold_df[feature_cols]
    )[:, 1]

    model_metadata["model_name"] = "logistic_regression_calendar_threshold_baseline"

else:
    warnings.warn("scikit-learn is unavailable. Falling back to smoothed month-threshold climatology.")

    smoothing_alpha = 5.0
    global_rate = train_df[target_col].mean()

    climatology = (
        train_df.groupby(["month", "threshold_c"], as_index=False)
        .agg(
            positive_count=(target_col, "sum"),
            observation_count=(target_col, "size"),
        )
    )

    climatology["baseline_classifier_probability"] = (
        climatology["positive_count"] + smoothing_alpha * global_rate
    ) / (
        climatology["observation_count"] + smoothing_alpha
    )

    historical_threshold_df = historical_threshold_df.merge(
        climatology[["month", "threshold_c", "baseline_classifier_probability"]],
        on=["month", "threshold_c"],
        how="left",
    )

    historical_threshold_df["baseline_classifier_probability"] = historical_threshold_df[
        "baseline_classifier_probability"
    ].fillna(global_rate)

    model_metadata["model_name"] = "smoothed_month_threshold_climatology"
    model_metadata["smoothing_alpha"] = smoothing_alpha

model_metadata

{'model_name': 'logistic_regression_calendar_threshold_baseline',
 'feature_cols': ['threshold_c',
  'day_of_year_sin',
  'day_of_year_cos',
  'month_sin',
  'month_cos'],
 'training_start_year': 2010,
 'validation_years': [2024, 2025],
 'test_years': [2026]}

## 8. Evaluate the baseline classifier

The validation and test periods are held out by calendar year. The test set may be short if the current-year HKO file is only available through part of the year, so it should be interpreted cautiously.

In [41]:
metric_rows = []

for split_name in ["train", "validation", "test"]:
    split_df = historical_threshold_df[historical_threshold_df["sample_split"] == split_name].copy()
    metric_rows.append(
        evaluate_binary_probabilities(
            split_df,
            target_col="realised_exceeds_threshold",
            prob_col="baseline_classifier_probability",
            label=split_name,
        )
    )

metrics_df = pd.DataFrame(metric_rows)
metrics_df

,sample,n_obs,positive_rate,brier_score,log_loss,roc_auc
0,train,51130,0.417524,0.083921,0.264747,0.955646
1,validation,7310,0.458550,0.098654,0.307104,0.945642
2,test,1510,0.263576,0.095637,0.294799,0.944848


In [43]:
def calibration_table(df, target_col, prob_col, n_bins=10):
    tmp = df[[target_col, prob_col]].dropna().copy()
    if tmp.empty:
        return pd.DataFrame()

    tmp["probability_bin"] = pd.qcut(
        tmp[prob_col].rank(method="first"),
        q=n_bins,
        labels=False,
        duplicates="drop",
    )

    return (
        tmp.groupby("probability_bin", as_index=False)
        .agg(
            n_obs=(target_col, "size"),
            mean_predicted_probability=(prob_col, "mean"),
            realised_frequency=(target_col, "mean"),
        )
    )

validation_calibration_df = calibration_table(
    historical_threshold_df[historical_threshold_df["sample_split"] == "validation"],
    target_col="realised_exceeds_threshold",
    prob_col="baseline_classifier_probability",
    n_bins=10,
)

validation_calibration_df

,probability_bin,n_obs,mean_predicted_probability,realised_frequency
0,0,731,0.000643,0.000000
1,1,731,0.004485,0.002736
2,2,731,0.019691,0.030096
3,3,731,0.065972,0.138167
4,4,731,0.175549,0.298222
5,5,731,0.402383,0.560876
6,6,731,0.666643,0.718194
7,7,731,0.873680,0.868673
8,8,731,0.968216,0.969904
9,9,731,0.995290,0.998632


## 9. Score the Step 7 Hong Kong market example

The Step 7 dataset contains the market-aligned threshold rows. This section adds baseline classifier probabilities to those rows.

The resulting comparison is descriptive because it is based on one market date. It is still useful because it checks that the baseline classifier can output probabilities on the same threshold grid used by Polymarket.

In [46]:
if step7_threshold_df.empty:
    step7_scored_df = pd.DataFrame()
    print("Step 7 threshold dataset not found. Run Notebook 9 first to generate market-aligned rows.")

else:
    step7_scored_df = step7_threshold_df.copy()
    step7_scored_df["target_local_date"] = pd.to_datetime(step7_scored_df["target_local_date"])
    step7_scored_df["year"] = step7_scored_df["target_local_date"].dt.year
    step7_scored_df["month"] = step7_scored_df["target_local_date"].dt.month
    step7_scored_df["day"] = step7_scored_df["target_local_date"].dt.day
    step7_scored_df["day_of_year"] = step7_scored_df["target_local_date"].dt.dayofyear
    step7_scored_df = add_calendar_features(step7_scored_df)

    if SKLEARN_AVAILABLE:
        step7_scored_df["baseline_classifier_probability"] = baseline_model.predict_proba(
            step7_scored_df[feature_cols]
        )[:, 1]
    else:
        step7_scored_df = step7_scored_df.merge(
            climatology[["month", "threshold_c", "baseline_classifier_probability"]],
            on=["month", "threshold_c"],
            how="left",
        )
        step7_scored_df["baseline_classifier_probability"] = step7_scored_df[
            "baseline_classifier_probability"
        ].fillna(global_rate)

    if "market_implied_threshold_prob_normalised" in step7_scored_df.columns:
        step7_scored_df["baseline_minus_market_probability"] = (
            step7_scored_df["baseline_classifier_probability"]
            - step7_scored_df["market_implied_threshold_prob_normalised"]
        )

    step7_scored_df["baseline_model_name"] = model_metadata["model_name"]

    display(step7_scored_df.head(40))

,contract_slug,city,target_local_date,settlement_source,settlement_variable,official_realised_temperature_c,threshold_c,realised_exceeds_threshold,forecast_model,forecast_run_time_utc,forecast_issue_time_utc,forecast_valid_time_utc,timing_label,forecast_timing_category,lead_time_to_valid_hours,lead_time_to_day_start_hours,lead_time_to_day_midpoint_hours,lead_time_to_day_end_hours,market_implied_threshold_prob_raw,market_implied_threshold_prob_normalised,number_of_contributing_bins,average_price_time_gap_minutes,market_price_timestamp_rule,forecast_temperature_c,forecast_temperature_source,forecast_feature_status,year,month,day,day_of_year,day_of_year_sin,day_of_year_cos,month_sin,month_cos,baseline_classifier_probability,baseline_minus_market_probability,baseline_model_name
0,highest-temperature-in-hong-kong-on-may-30-2026,Hong Kong,2026-05-30,Hong Kong Observatory,Daily Maximum Temperature,32.6,24.0,1,AIFS_or_ECMWF_example,2026-05-28 12:00:00+00:00,2026-05-28 12:00:00+00:00,2026-05-30 12:00:00+00:00,two_day_valid_time_example,full_day_ex_ante,48.0,28.0,40.0,51.999722,0.9645,0.998964,10,0.100000,first available Polymarket price at or after f...,NaN,not_attached_in_step_7,pending_ai_weather_feature_merge,2026,5,30,150,0.536696,-0.843776,0.5,-0.866025,0.992736,-0.006229,logistic_regression_calendar_threshold_baseline
1,highest-temperature-in-hong-kong-on-may-30-2026,Hong Kong,2026-05-30,Hong Kong Observatory,Daily Maximum Temperature,32.6,25.0,1,AIFS_or_ECMWF_example,2026-05-28 12:00:00+00:00,2026-05-28 12:00:00+00:00,2026-05-30 12:00:00+00:00,two_day_valid_time_example,full_day_ex_ante,48.0,28.0,40.0,51.999722,0.9620,0.996375,9,0.100000,first available Polymarket price at or after f...,NaN,not_attached_in_step_7,pending_ai_weather_feature_merge,2026,5,30,150,0.536696,-0.843776,0.5,-0.866025,0.984324,-0.012050,logistic_regression_calendar_threshold_baseline
2,highest-temperature-in-hong-kong-on-may-30-2026,Hong Kong,2026-05-30,Hong Kong Observatory,Daily Maximum Temperature,32.6,26.0,1,AIFS_or_ECMWF_example,2026-05-28 12:00:00+00:00,2026-05-28 12:00:00+00:00,2026-05-30 12:00:00+00:00,two_day_valid_time_example,full_day_ex_ante,48.0,28.0,40.0,51.999722,0.9595,0.993786,8,0.100000,first available Polymarket price at or after f...,NaN,not_attached_in_step_7,pending_ai_weather_feature_merge,2026,5,30,150,0.536696,-0.843776,0.5,-0.866025,0.966503,-0.027283,logistic_regression_calendar_threshold_baseline
3,highest-temperature-in-hong-kong-on-may-30-2026,Hong Kong,2026-05-30,Hong Kong Observatory,Daily Maximum Temperature,32.6,27.0,1,AIFS_or_ECMWF_example,2026-05-28 12:00:00+00:00,2026-05-28 12:00:00+00:00,2026-05-30 12:00:00+00:00,two_day_valid_time_example,full_day_ex_ante,48.0,28.0,40.0,51.999722,0.9540,0.988089,7,0.100000,first available Polymarket price at or after f...,NaN,not_attached_in_step_7,pending_ai_weather_feature_merge,2026,5,30,150,0.536696,-0.843776,0.5,-0.866025,0.929862,-0.058227,logistic_regression_calendar_threshold_baseline
4,highest-temperature-in-hong-kong-on-may-30-2026,Hong Kong,2026-05-30,Hong Kong Observatory,Daily Maximum Temperature,32.6,28.0,1,AIFS_or_ECMWF_example,2026-05-28 12:00:00+00:00,2026-05-28 12:00:00+00:00,2026-05-30 12:00:00+00:00,two_day_valid_time_example,full_day_ex_ante,48.0,28.0,40.0,51.999722,0.9505,0.984464,6,0.100000,first available Polymarket price at or after f...,NaN,not_attached_in_step_7,pending_ai_weather_feature_merge,2026,5,30,150,0.536696,-0.843776,0.5,-0.866025,0.858992,-0.125472,logistic_regression_calendar_threshold_baseline
5,highest-temperature-in-hong-kong-on-may-30-2026,Hong Kong,2026-05-30,Hong Kong Observatory,Daily Maximum Temperature,32.6,29.0,1,AIFS_or_ECMWF_example,2026-05-28 12:00:00+00:00,2026-05-28 12:00:00+00:00,2026-05-30 12:00:00+00:00,two_day_valid_time_example,full_day_ex_ante,48.0,28.0,40.0,51.999722,0.9155,0.948213,5,0.100000,first available Polymarket price at or after f...,NaN,not_attached_in_step_7,pending_ai_weather_feature_merge,2026,5,30,150,0.536696

In [48]:
step7_metrics_rows = []

if not step7_scored_df.empty:
    step7_metrics_rows.append(
        evaluate_binary_probabilities(
            step7_scored_df,
            target_col="realised_exceeds_threshold",
            prob_col="baseline_classifier_probability",
            label="step7_baseline_classifier",
        )
    )

    if "market_implied_threshold_prob_normalised" in step7_scored_df.columns:
        step7_metrics_rows.append(
            evaluate_binary_probabilities(
                step7_scored_df,
                target_col="realised_exceeds_threshold",
                prob_col="market_implied_threshold_prob_normalised",
                label="step7_market_implied_probability",
            )
        )

step7_metrics_df = pd.DataFrame(step7_metrics_rows)
step7_metrics_df

,sample,n_obs,positive_rate,brier_score,log_loss,roc_auc
0,step7_baseline_classifier,40,0.9,0.131270,0.381431,1.0
1,step7_market_implied_probability,40,0.9,0.052922,0.162932,1.0


## 10. Save outputs

In [51]:
historical_training_output_path = PROCESSED_DIR / "hko_historical_threshold_training_dataset.csv"
metrics_output_path = PROCESSED_DIR / "hko_baseline_classifier_metrics.csv"
calibration_output_path = PROCESSED_DIR / "hko_baseline_classifier_validation_calibration.csv"
step7_predictions_output_path = PROCESSED_DIR / "hong_kong_step7_baseline_classifier_predictions.csv"

historical_threshold_df.to_csv(historical_training_output_path, index=False)
metrics_df.to_csv(metrics_output_path, index=False)
validation_calibration_df.to_csv(calibration_output_path, index=False)

if not step7_scored_df.empty:
    step7_scored_df.to_csv(step7_predictions_output_path, index=False)

print("Saved historical training dataset:", historical_training_output_path)
print("Saved metrics:", metrics_output_path)
print("Saved validation calibration:", calibration_output_path)

if not step7_scored_df.empty:
    print("Saved Step 7 baseline predictions:", step7_predictions_output_path)
else:
    print("No Step 7 predictions saved because the Step 7 dataset was not found.")

Saved historical training dataset: ../data/processed/baseline_classifier/hko_historical_threshold_training_dataset.csv
Saved metrics: ../data/processed/baseline_classifier/hko_baseline_classifier_metrics.csv
Saved validation calibration: ../data/processed/baseline_classifier/hko_baseline_classifier_validation_calibration.csv
Saved Step 7 baseline predictions: ../data/processed/baseline_classifier/hong_kong_step7_baseline_classifier_predictions.csv


## 11. Interpretation

This notebook creates the first baseline threshold classifier for the project.

The model is intentionally simple. It uses official Hong Kong Observatory historical daily maximum temperatures, calendar features and the threshold level to estimate the probability that daily maximum temperature exceeds a given threshold. This creates a transparent benchmark before AI weather forecast features are added.

The baseline classifier is not the final model. It does not yet condition on AI weather forecasts, ensemble information, forecast lead time or current atmospheric state. Its role is to provide a supervised-learning scaffold, a climatological benchmark and a probability-output format that can be compared with Polymarket prices.

The next step is to merge AI or weather-model forecast features at the same forecast issue times. Once those features are available, the classifier can be extended from a climatological benchmark into a forecast-informed post-processing model.